# Primer Parcial

Predecir si un alumno puede aprobar en uno cualquiera de los finales que rinde sabiendo su firma.

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 1000)

csv_path = 'reglamento_nuevo_unificado.csv'
if not os.path.exists(csv_path):
    csv_path = os.path.join('..', 'Clase_5', 'reglamento_nuevo_unificado.csv')

df_raw = pd.read_csv(csv_path)
print(f"Dataset cargado exitosamente: {df_raw.shape[0]:,} filas y {df_raw.shape[1]} columnas.")


Dataset cargado exitosamente: 64,295 filas y 29 columnas.


# Mapeo de las carreras


In [ ]:
career_code= { 

    'codes': ['CIV-PLS13','CIV-PLS23','INT9CONSTR','INT9TRANSP','INT9ORTERR','INT9SANEHI','ELE-PLS13'
    ,'ELE-PLS23','INT9ELECTR','INT9SDIGYT','MCT-PLS13','MCT-PLS23','MCT9-OPT','IND-PLS13','IND-PLS23'
    ,'INT9G-ECO','INT9-PROYT','CGF-PLS13','CGF-PLS23','INT9RNYMA','MEC-PLS13','MEC-PLS23','INT9MECANI'
    ,'MEC9-OPT','ECA-PLS13','ECA-PLS23','ECA9-OPT']

}

career_code_mapping = {
    
    'CIV-PLS13': 'Ing. Civil', 'CIV-PLS23': 'Ing. Civil', 'INT9CONSTR': 'Ing. Civil',
    'INT9TRANSP': 'Ing. Civil', 'INT9ORTERR': 'Ing. Civil', 'INT9SANEHI': 'Ing. Civil',
    
    'ELE-PLS13': 'Ing. Electromecánica', 'ELE-PLS23': 'Ing. Electromecánica',
    'INT9ELECTR': 'Ing. Electromecánica', 'INT9SDIGYT': 'Ing. Electromecánica',
    
    'MCT-PLS13': 'Ing. Mecatrónica', 'MCT-PLS23': 'Ing. Mecatrónica', 'MCT9-OPT': 'Ing. Mecatrónica',
    
    'IND-PLS13': 'Ing. Industrial', 'IND-PLS23': 'Ing. Industrial',
    'INT9G-ECO': 'Ing. Industrial', 'INT9-PROYT': 'Ing. Industrial',
    
    'CGF-PLS13': 'Ing. Geográfica', 'CGF-PLS23': 'Ing. Geográfica', 'INT9RNYMA': 'Ing. Geográfica',
    
    'MEC-PLS13': 'Ing. Mecánica', 'MEC-PLS23': 'Ing. Mecánica',
    'INT9MECANI': 'Ing. Mecánica', 'MEC9-OPT': 'Ing. Mecánica',
    
    'ECA-PLS13': 'Ing. Electrónica', 'ECA-PLS23': 'Ing. Electrónica', 'ECA9-OPT': 'Ing. Electrónica'
}

df_clean=df_raw.copy()

# 2. Mapeo de la columna 'Firma'
df_clean['Firma_Total']=df_clean['Firma'].astype(str).str.strip().map(career_code_mapping)

# 3. Limpieza y definición del Target, con esto extraemos la última nota 
df_clean['Nota_Num'] = (df_clean['Nota.Final'].astype(str).str.extractall(r'(\d+)')[0].groupby(level=0).last().astype(float))

# Mantener NaN si originalmente no rindió final
df_clean.loc[df_clean['Nota.Final'].isna(), 'Nota_Num'] = np.nan

# Target binario: 1 si aprobó (nota >= 2), 0 si reprobó (nota = 1) o no rindió (NaN)
df_clean['Target'] = (df_clean['Nota_Num'] >= 2).astype(int)

print("Target creado y listo para el análisis")

Filas tras mapeo de carreras: 64,295


# Limpieza de los datos y obtención de información
Filtrado y conversión a datos tipos enteros o flotantes. Para este caso vamos a analizar el puntaje para firma y vamos a estudiar los datos anteriores si es más factible rendir un primer final o un segundo final.